# WP4v4 — Notebook 1 : Génération de la base de données (v4b)

**Nouveauté** : K=10 tokens CLS MAE masqués par image (seeds 0..9)
associés au même CLS CLIP.

- 130k images × 10 masquages = **1.3M paires** d'entraînement
- Chaque CLS MAE masqué capture l'essence sémantique depuis un sous-ensemble
  différent de patches (confirmé par les UMAP qui clusterisent bien)
- La même cible CLS CLIP force f_theta à apprendre une projection
  invariante au masquage -> meilleure généralisation aux patch tokens individuels

In [ ]:
import torch
import torch.nn.functional as F
from transformers import ViTImageProcessor, ViTMAEModel
from transformers import LlavaForConditionalGeneration, CLIPImageProcessor
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
K_SAMPLINGS = 10  # tirages masques par image
print(f'Device : {DEVICE} | K_SAMPLINGS : {K_SAMPLINGS}')


In [ ]:
mae_processor = ViTImageProcessor(
    size={'height': 224, 'width': 224},
    image_mean=[0.485, 0.456, 0.406],
    image_std=[0.229, 0.224, 0.225],
)
mae_encoder = ViTMAEModel.from_pretrained('./vit-mae-large').to(DEVICE)
mae_encoder.eval()

llava = LlavaForConditionalGeneration.from_pretrained(
    './llava-1.5-7b-hf', torch_dtype=torch.float16
)
vision_tower = llava.vision_tower.to(DEVICE).eval()
del llava; torch.cuda.empty_cache()
clip_proc = CLIPImageProcessor.from_pretrained('./llava-1.5-7b-hf')
print(f'Modeles charges — VRAM : {torch.cuda.memory_allocated()/1e9:.1f} GB')


In [ ]:
ds_train = load_dataset('parquet', data_files={'train': './imagenet100/data/train-*.parquet'})
ds_val   = load_dataset('parquet', data_files={'validation': './imagenet100/data/validation-*.parquet'})

transform = transforms.Compose([
    transforms.Resize(384), transforms.CenterCrop(336), transforms.ToTensor(),
])

class HFImageDataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.data = hf_dataset; self.transform = transform
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        img  = item['image'].convert('RGB')
        if self.transform: img = self.transform(img)
        return img, item['label']

dataset_train = HFImageDataset(ds_train['train'],    transform=transform)
dataset_val   = HFImageDataset(ds_val['validation'], transform=transform)
print(f'Train : {len(dataset_train)} images -> {len(dataset_train)*K_SAMPLINGS} paires attendues')
print(f'Val   : {len(dataset_val)} images -> {len(dataset_val)*K_SAMPLINGS} paires attendues')


In [ ]:
def generate_pairs(dataset, batch_size=64, desc=''):
    """
    Pour chaque image, genere K tokens CLS MAE masques (seeds 0..K-1)
    associes au meme CLS CLIP (calcule une seule fois).

    Retourne des donnees BRUTES (non normalisees) :
    - cls_mae  : (N*K, 1024) — K tokens CLS MAE par image
    - cls_clip : (N*K, 1024) — K fois le meme CLS CLIP par image, L2-normalise
    - labels   : (N*K,)
    """
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=0, pin_memory=(DEVICE=='cuda'))
    all_cls_mae, all_cls_clip, all_labels = [], [], []

    for images_336, labels in tqdm(loader, desc=desc):
        B = images_336.shape[0]
        images_224 = F.interpolate(images_336, size=(224,224),
                                   mode='bilinear', align_corners=False)

        # CLS CLIP — calcule une seule fois par image (pas de randomness)
        clip_in = clip_proc(images=list(images_336), return_tensors='pt', do_rescale=False)
        pix = clip_in['pixel_values'].to(DEVICE).half()
        with torch.no_grad():
            cls_clip_batch = vision_tower(pix).last_hidden_state[:, 0]  # (B, 1024)
        cls_clip_batch = F.normalize(cls_clip_batch.cpu().float(), dim=-1)  # L2-normalise

        # MAE inputs (communs a tous les seeds)
        mae_in = mae_processor(images=list(images_224), return_tensors='pt', do_rescale=False)
        mae_in = {k: v.to(DEVICE) for k, v in mae_in.items()}

        # K tokens CLS MAE masques par image
        for seed in range(K_SAMPLINGS):
            gen   = torch.Generator().manual_seed(seed)
            noise = torch.rand(B, 196, generator=gen).to(DEVICE)
            with torch.no_grad():
                out = mae_encoder(**mae_in, noise=noise)
            cls_mae_batch = out.last_hidden_state[:, 0].cpu().float()  # (B, 1024)

            all_cls_mae.append(cls_mae_batch)
            all_cls_clip.append(cls_clip_batch)   # meme cible pour tous les seeds
            all_labels.append(labels)

    return {
        'cls_mae':  torch.cat(all_cls_mae),   # (N*K, 1024) brut
        'cls_clip': torch.cat(all_cls_clip),  # (N*K, 1024) L2-normalise
        'labels':   torch.cat(all_labels),    # (N*K,)
    }

data_train = generate_pairs(dataset_train, batch_size=64, desc='Train')
data_val   = generate_pairs(dataset_val,   batch_size=64, desc='Val')

torch.save(data_train, 'wp4v4_pairs_train.pt')
torch.save(data_val,   'wp4v4_pairs_val.pt')
print(f'Sauvegarde OK')
print(f'  Train : {data_train["cls_mae"].shape}')  # (N*K, 1024)
print(f'  Val   : {data_val["cls_mae"].shape}')


In [ ]:
# Stats de normalisation — calculees sur train uniquement
cls_mean = data_train['cls_mae'].mean(dim=0)
cls_std  = data_train['cls_mae'].std(dim=0).clamp(min=1e-6)
torch.save({'mean': cls_mean, 'std': cls_std}, 'wp4v4_cls_norm_stats.pt')
print('Stats de normalisation sauvegardees')
print(f'  cls_mean norme : {cls_mean.norm():.3f}')
print(f'  cls_std  norme : {cls_std.norm():.3f}')

# Verification : les CLS masques d'une meme image sont-ils proches ?
# Prendre les K=10 premiers tokens (premier batch d'images, 10 seeds)
N = len(dataset_train)
cls_image0 = torch.stack([
    data_train['cls_mae'][seed * (N // batch_size * batch_size)]  # approximation
    for seed in range(K_SAMPLINGS)
][:5])  # 5 seeds pour la lisibilite

cls_image0_norm = F.normalize(cls_image0, dim=-1)
sim = (cls_image0_norm @ cls_image0_norm.T).numpy()
print('\nSimilarites cosinus entre 5 tirages masques de la meme image :')
print(sim.round(4))
print('(valeurs ~0.88-0.95 attendues — discriminant mais pas identique)')

# Verification discriminabilite inter-images
idx5 = [i * K_SAMPLINGS for i in range(5)]  # seed=0 de 5 images differentes
vecs = F.normalize(data_val['cls_mae'][idx5], dim=-1)
sim_inter = (vecs @ vecs.T).numpy()
print('\nSimilarites cosinus inter-images (seed=0) :')
print(sim_inter.round(4))

del vision_tower, mae_encoder; torch.cuda.empty_cache()
